In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")


DATA_PATH = Path("transactions.csv")
MODEL_PATH = Path("subscription_model.joblib")
SUBSCRIPTIONS_OUTPUT_PATH = Path("subscriptions_detected.csv")

RANDOM_STATE = 42

In [ ]:
df = pd.read_csv(DATA_PATH, sep=";")
df.columns = [c.strip().lower() for c in df.columns]

required_columns = {"user_id", "amount",
                    "mcc_code", "merchant_name", "transaction_date"}
missing = required_columns - set(df.columns)
if "transaction_time" not in df.columns:
    df["transaction_time"] = "00:00:00"
if "operation_type" not in df.columns:
    df["operation_type"] = "payment"
if "currency" not in df.columns:
    df["currency"] = "RUB"

df["merchant_name"] = df["merchant_name"].astype(str).str.strip()
df["mcc_code"] = df["mcc_code"].astype(str).str.strip()
df["operation_type"] = df["operation_type"].astype(str).str.lower().str.strip()
df["currency"] = df["currency"].astype(str).str.upper().str.strip()
df["amount"] = df["amount"].astype(str).str.replace(
    ",", ".", regex=False).astype(float).abs()
df["transaction_date"] = pd.to_datetime(
    df["transaction_date"], errors="coerce")

df = df[df["operation_type"].isin(
    ["payment", "purchase", "card_payment", "withdrawal", "debit"])].copy()
df.head()

In [ ]:
SUBSCRIPTION_MCC = {
    "4899", "4816", "5734", "5815", "5816", "5817", "5818", "5968", "7994", "8299",
}

def normalize_merchant(name: str) -> str:
    name = str(name).lower().strip()
    for ch in ["*", "_", "-", ".", ",", "  "]:
        name = name.replace(ch, " ")
    return " ".join(name.split())

df["merchant_norm"] = df["merchant_name"].apply(normalize_merchant)
group_cols = ["user_id", "merchant_norm", "mcc_code", "currency"]

def build_group_features(group: pd.DataFrame) -> pd.Series:
    group = group.sort_values("transaction_date")
    dates = group["transaction_date"].drop_duplicates().sort_values()
    amounts = group["amount"].astype(float)

    if len(dates) >= 2:
        intervals = dates.diff().dt.days.dropna()
        mean_interval = float(intervals.mean())
        std_interval = float(intervals.std(ddof=0)) if len(intervals) > 1 else 0.0
        min_interval = float(intervals.min())
        max_interval = float(intervals.max())
    else:
        mean_interval = np.nan
        std_interval = np.nan
        min_interval = np.nan
        max_interval = np.nan

    amount_mean = float(amounts.mean())
    amount_std = float(amounts.std(ddof=0)) if len(amounts) > 1 else 0.0
    amount_cv = amount_std / amount_mean if amount_mean > 0 else 0.0

    first_date = dates.min()
    last_date = dates.max()
    active_days = int((last_date - first_date).days) if len(dates) else 0

    return pd.Series({
        "user_id": group["user_id"].iloc[0],
        "merchant_name": group["merchant_name"].mode().iloc[0],
        "merchant_norm": group["merchant_norm"].iloc[0],
        "mcc_code": str(group["mcc_code"].iloc[0]),
        "currency": group["currency"].iloc[0],
        "transaction_count": int(len(group)),
        "unique_payment_days": int(len(dates)),
        "amount_mean": amount_mean,
        "amount_std": amount_std,
        "amount_cv": amount_cv,
        "mean_interval": mean_interval,
        "std_interval": std_interval,
        "min_interval": min_interval,
        "max_interval": max_interval,
        "active_days": active_days,
        "first_payment_date": first_date,
        "last_payment_date": last_date,
        "merchant_text": group["merchant_norm"].iloc[0],
        "mcc_is_subscription_like": int(str(group["mcc_code"].iloc[0]) in SUBSCRIPTION_MCC),
    })

features = df.groupby(group_cols, dropna=False).apply(build_group_features).reset_index(drop=True)
features.head()